In [1]:
import requests
import pandas as pd
from time import sleep
import os

def descargar_datos_ivia_completo(años):
    url = "http://riegos.ivia.es/ajax/calculos.php"
    headers = {
        'Content-Type': 'application/x-www-form-urlencoded',
        'Referer': 'http://riegos.ivia.es/datos-meteorologicos'
    }
    
    # Mapeo de nombres REALES de la API a los nombres ESTÁNDAR
    rename_map = {
        'tmedia': 'temp', 'hmedia': 'humedad', 'radiacion': 'radiacion',
        'precipitacion': 'precipitacion', 'viento': 'viento', 'direccion': 'direccion'
    }
    
    # Lista de columnas que queremos conservar después del renombrado
    columnas_finales_semi = ['temp', 'humedad', 'radiacion', 'precipitacion', 'fecha_hora', 'fecha_dia']
    
    todos_los_datos = []

    # --- PASO 1: Descargar datos SEMIHORARIOS ---
    print("--- FASE 1: Descargando datos SEMIHORARIOS ---")
    
    for año in años:
        print(f"  -> Año {año} (semihorario)...")
        data = {
            't': 'meteorologia', 'r': 'semihorarios', 'e[0][p]': '3', 'e[0][e]': '21',
            'i[y]': str(año), 'i[m]': '1', 'i[d]': '1',
            'f[y]': str(año), 'f[m]': '12', 'f[d]': '31',
        }
        
        try:
            resp = requests.post(url, data=data, headers=headers)
            resp.raise_for_status()
            json_data = resp.json()
            
            if 'data' in json_data and json_data['data']:
                df = pd.DataFrame(json_data['data'])
                
                # APLICAR RENOMBRADO (tmedia -> temp, hmedia -> humedad)
                df = df.rename(columns=rename_map)

                # 1. Corrección Fecha+Hora y Error 24:00
                if 'hora' in df.columns:
                    mask_24 = df['hora'] == '24:00'
                    df.loc[mask_24, 'hora'] = '00:00'
                    df['temp_dt'] = pd.to_datetime(df['fecha'] + ' ' + df['hora'], dayfirst=True)
                    df.loc[mask_24, 'temp_dt'] += pd.Timedelta(days=1)
                    df['fecha_hora'] = df['temp_dt']
                    df['fecha_dia'] = df['fecha_hora'].dt.normalize()
                
                # 2. Filtrar columnas útiles (usando los nombres estándar 'temp', 'humedad')
                cols_existentes = [c for c in columnas_finales_semi if c in df.columns]
                df = df[cols_existentes]
                
                todos_los_datos.append(df)
                
            else:
                print(f"  -> No hay datos para el año {año}")
                
        except Exception as e: print(f"Error en año {año}: {e}")
        sleep(1)

    if not todos_los_datos: return None
    df_semi_total = pd.concat(todos_los_datos, ignore_index=True)

    # --- PASO 2: Descargar datos DIARIOS (solo ETo) ---
    print("\n--- FASE 2: Descargando datos DIARIOS (para ETo) ---")
    dfs_diario = []
    
    for año in años:
        print(f"  -> Año {año} (Diario)...")
        data_diaria = {
            't': 'meteorologia', 'r': 'diarios', 'e[0][p]': '3', 'e[0][e]': '21',
            'i[y]': str(año), 'i[m]': '1', 'i[d]': '1', 'f[y]': str(año), 'f[m]': '12', 'f[d]': '31',
        }
        try:
            resp = requests.post(url, data=data_diaria, headers=headers)
            if resp.ok and 'data' in resp.json() and resp.json()['data']:
                df = pd.DataFrame(resp.json()['data'])
                
                if 'eto' in df.columns:
                    df['fecha_dia'] = pd.to_datetime(df['fecha'], dayfirst=True)
                    dfs_diario.append(df[['fecha_dia', 'eto']])
        except: pass
        sleep(1)

    if not dfs_diario: 
        print("No se ha descargado ETo.")
        return df_semi_total

    df_eto_total = pd.concat(dfs_diario, ignore_index=True)

    # --- PASO 3: FUSIÓN (Merge) ---
    
    # Merge por 'fecha_dia'
    df_final = pd.merge(df_semi_total, df_eto_total, on='fecha_dia', how='left')
    
    # Limpieza final
    df_final = df_final.drop(columns=['fecha_dia'])
    df_final = df_final.rename(columns={'fecha_hora': 'fecha'})
    
    # Ordenamos y reseteamos índice
    df_final = df_final.sort_values('fecha').reset_index(drop=True)

    # Reordenar columnas para que 'fecha' esté primero
    cols = ['fecha'] + [c for c in df_final.columns if c != 'fecha']
    df_final = df_final[cols]

    # Renombrado de columnas
    nombres = {'fecha':'instante','temp': 'TA', 'humedad': 'HA'}
    df_final.rename(columns=nombres, inplace=True)

    return df_final


if __name__ == "__main__":
    años = [2018,2019, 2021, 2022, 2023]
    df_completo = descargar_datos_ivia_completo(años)

    ruta = 'meteo_ivia.csv'
    carpeta = os.path.dirname(ruta)
    if carpeta and not os.path.exists(carpeta):
        os.makedirs(carpeta)

    df_completo.to_csv(ruta, index=False, date_format='%Y-%m-%d %H:%M:%S')
    print(f"\nArchivo guardado en: {ruta}")
    print(df_completo.head())
    print("Columnas:", df_completo.columns.tolist())

--- FASE 1: Descargando datos SEMIHORARIOS ---
  -> Año 2018 (semihorario)...
  -> Año 2019 (semihorario)...
  -> Año 2021 (semihorario)...
  -> Año 2022 (semihorario)...
  -> Año 2023 (semihorario)...

--- FASE 2: Descargando datos DIARIOS (para ETo) ---
  -> Año 2018 (Diario)...
  -> Año 2019 (Diario)...
  -> Año 2021 (Diario)...
  -> Año 2022 (Diario)...
  -> Año 2023 (Diario)...

Archivo guardado en: meteo_ivia.csv
             instante     TA     HA  radiacion  precipitacion   eto
0 2018-01-01 00:30:00  15.32  38.46        0.0            0.0  2.96
1 2018-01-01 01:00:00  14.81  37.61        0.0            0.0  2.96
2 2018-01-01 01:30:00  14.53  39.27        0.0            0.0  2.96
3 2018-01-01 02:00:00  14.33  38.63        0.0            0.0  2.96
4 2018-01-01 02:30:00  13.80  36.77        0.0            0.0  2.96
Columnas: ['instante', 'TA', 'HA', 'radiacion', 'precipitacion', 'eto']


In [2]:
import pandas as  pd
df = pd.read_csv("meteo_ivia.csv",index_col='instante')
df.head(48)

,TA,HA,radiacion,precipitacion,eto
instante,,,,,
2018-01-01 00:30:00,15.32,38.46,0.00,0.0,2.96
2018-01-01 01:00:00,14.81,37.61,0.00,0.0,2.96
2018-01-01 01:30:00,14.53,39.27,0.00,0.0,2.96
2018-01-01 02:00:00,14.33,38.63,0.00,0.0,2.96
2018-01-01 02:30:00,13.80,36.77,0.00,0.0,2.96
2018-01-01 03:00:00,13.62,35.21,0.00,0.0,2.96
2018-01-01 03:30:00,13.31,36.48,0.00,0.0,2.96
2018-01-01 04:00:00,12.81,39.08,0.00,0.0,2.96
2018-01-01 04:30:00,12.70,38.45,0.00,0.0,2.96
